# Visualização Local — lh_dados_publicos · SQL Endpoint

Conecta ao Fabric via SQLAlchemy + ActiveDirectoryInteractive.

**Rode em ordem:** Célula 1 → Célula 2 → demais visuals.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import urllib.parse
from sqlalchemy import create_engine, text

SERVER   = 'ena6obg6j2cevcppw7dn7yu57a-knnp5frchjbujdik4l3nmgrgsa.datawarehouse.fabric.microsoft.com'
DATABASE = 'lh_dados_publicos'

conn_str = (
    f'Driver={{ODBC Driver 18 for SQL Server}};'
    f'Server={SERVER};'
    f'Database={DATABASE};'
    'Authentication=ActiveDirectoryInteractive;'
    'Encrypt=yes;'
)
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(
    f'mssql+pyodbc:///?odbc_connect={params}',
    isolation_level='AUTOCOMMIT'
)

def sql(query):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

CORES = {'Santos': '#1f77b4', 'Osasco': '#ff7f0e', 'Maua': '#2ca02c'}

print('[OK] Imports e engine prontos. Autenticacao abre ao executar a primeira query.')

In [ ]:
# Mapeia todas as tabelas do lakehouse — execute ANTES das queries
df_disco = sql("""
    SELECT TABLE_SCHEMA AS schema_,
           TABLE_NAME   AS tabela,
           TABLE_SCHEMA + '.' + TABLE_NAME AS nome_completo
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")

print(f'Total de tabelas no lakehouse: {len(df_disco)}')
display(df_disco)

# Lookup rapido: 'pib' -> 'silver.pib', 'silver.pib' -> 'silver.pib'
TB = {}
for _, r in df_disco.iterrows():
    TB[r.tabela]      = r.nome_completo
    TB[r.nome_completo] = r.nome_completo

def T(nome):
    """Retorna nome completo da tabela com schema. Lanca KeyError informativo se nao existir."""
    if nome in TB:
        return TB[nome]
    # Tenta variantes comuns
    for v in [f'silver.{nome}', f'gold.{nome}', f'dbo.{nome}',
              nome.replace('.','_'), nome.replace('_','.', 1)]:
        if v in TB:
            return TB[v]
    raise KeyError(
        f"Tabela '{nome}' nao existe no lakehouse.\n"
        f"Tabelas disponiveis: {sorted(TB.keys())}"
    )

# Status das tabelas gold
gold_esperadas = [
    'gold_mercado_trabalho', 'gold_populacao_municipios', 'gold_pib_municipios'
]
print('\n--- Status tabelas gold ---')
for g in gold_esperadas:
    existe = any(g in k for k in TB)
    status = 'OK' if existe else 'NAO CRIADA — rode o notebook gold no Fabric'
    print(f'  {g}: {status}')

## 1. PIB per Capita — `silver.pib`

In [ ]:
df_pib = sql(f"""
    SELECT nome_municipio, ano,
           valor AS pib_per_capita
    FROM {T('pib')}
    WHERE indicador = 'pib_per_capita_r'
    ORDER BY ano, nome_municipio
""")
print(f'Linhas: {len(df_pib)} | Municipios: {df_pib.nome_municipio.nunique()} | Anos: {sorted(df_pib.ano.unique())[:5]}')
display(df_pib.head())

fig = px.line(
    df_pib, x='ano', y='pib_per_capita', color='nome_municipio', markers=True,
    title='Evolucao PIB per Capita por Municipio',
    labels={'ano': 'Ano', 'pib_per_capita': 'PIB per Capita (R$)', 'nome_municipio': 'Municipio'}
)
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 2. Taxa de Formalização — `silver.rais` × `silver.populacao` (2022)

> Usa silver diretamente. Coluna RAIS: `quantidade_vinculos_ativos`.

In [ ]:
try:
    rais = T('rais')
    pop  = T('populacao')
    df_formal = sql(f"""
        WITH r AS (
            SELECT id_municipio, nome_municipio, cluster,
                   SUM(CAST(quantidade_vinculos_ativos AS FLOAT)) AS vinculos
            FROM {rais}
            WHERE ano = 2022
            GROUP BY id_municipio, nome_municipio, cluster
        ),
        p AS (
            SELECT id_municipio,
                   SUM(TRY_CAST(valor AS FLOAT)) AS populacao
            FROM {pop}
            WHERE ano = 2022 AND indicador = 'populacao_residente'
            GROUP BY id_municipio
        )
        SELECT r.nome_municipio, r.cluster,
               r.vinculos, p.populacao,
               ROUND((r.vinculos / NULLIF(p.populacao,0)) * 100, 2) AS pct_formalizacao
        FROM r JOIN p ON r.id_municipio = p.id_municipio
        ORDER BY pct_formalizacao DESC
    """)
    display(df_formal)
    fig = px.bar(
        df_formal.sort_values('pct_formalizacao'),
        x='pct_formalizacao', y='nome_municipio',
        color='cluster', color_discrete_map=CORES,
        orientation='h', text='pct_formalizacao',
        title='Taxa de Formalizacao por Municipio — RAIS vs Populacao (2022)',
        labels={'pct_formalizacao': '% Formalizacao', 'nome_municipio': 'Municipio'}
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_white', xaxis_ticksuffix='%', height=600)
    fig.show()
except KeyError as e:
    print(f'TABELA NAO ENCONTRADA:\n{e}')

## 3. Saldo CAGED por Cluster — `silver.caged` (2020–2025)

> Coluna silver: `saldo_movimentacao`.

In [ ]:
try:
    caged = T('caged')
    df_caged = sql(f"""
        SELECT cluster, ano,
               SUM(CAST(saldo_movimentacao AS FLOAT)) AS saldo_anual
        FROM {caged}
        WHERE ano BETWEEN 2020 AND 2025
        GROUP BY cluster, ano
        ORDER BY cluster, ano
    """)
    display(df_caged)
    fig = px.line(
        df_caged, x='ano', y='saldo_anual', color='cluster',
        color_discrete_map=CORES, markers=True,
        title='Saldo CAGED Anual por Cluster — 2020-2025',
        labels={'ano': 'Ano', 'saldo_anual': 'Saldo', 'cluster': 'Cluster'}
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Equilibrio')
    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()
except KeyError as e:
    print(f'TABELA NAO ENCONTRADA:\n{e}')

## 4. Top 5 Setores CNAE por Cluster — `silver.rais` (2022)

> Coluna silver: `cnae_2`.

In [ ]:
try:
    rais = T('rais')
    df_cnae = sql(f"""
        SELECT cluster, cnae_2 AS secao_cnae,
               SUM(CAST(quantidade_vinculos_ativos AS FLOAT)) AS vinculos
        FROM {rais}
        WHERE ano = 2022 AND cnae_2 IS NOT NULL
        GROUP BY cluster, cnae_2
    """)
    top5 = (
        df_cnae.sort_values('vinculos', ascending=False)
               .groupby('cluster').head(5)
               .reset_index(drop=True)
    )
    display(top5)
    fig = px.bar(
        top5.sort_values('vinculos'),
        x='vinculos', y='secao_cnae', color='cluster',
        facet_col='cluster', color_discrete_map=CORES,
        orientation='h',
        title='Top 5 Secoes CNAE por Cluster — RAIS 2022',
        labels={'vinculos': 'Vinculos ativos', 'secao_cnae': 'CNAE'}
    )
    fig.update_layout(template='plotly_white', showlegend=False, height=450)
    fig.show()
except KeyError as e:
    print(f'TABELA NAO ENCONTRADA:\n{e}')

## 5. Scatter Formalização × População — JOIN por `id_municipio`

In [ ]:
try:
    rais = T('rais')
    pop  = T('populacao')
    df_scatter = sql(f"""
        WITH r AS (
            SELECT id_municipio, nome_municipio, cluster,
                   SUM(CAST(quantidade_vinculos_ativos AS FLOAT)) AS empregos
            FROM {rais}
            WHERE ano = 2022
            GROUP BY id_municipio, nome_municipio, cluster
        ),
        p AS (
            SELECT id_municipio,
                   SUM(TRY_CAST(valor AS FLOAT)) AS populacao
            FROM {pop}
            WHERE ano = 2022 AND indicador = 'populacao_residente'
            GROUP BY id_municipio
        )
        SELECT r.nome_municipio, r.cluster,
               p.populacao, r.empregos,
               ROUND((r.empregos / NULLIF(p.populacao,0)) * 100, 2) AS pct_formalizacao
        FROM r JOIN p ON r.id_municipio = p.id_municipio
    """)
    display(df_scatter)
    fig = px.scatter(
        df_scatter, x='populacao', y='pct_formalizacao',
        color='cluster', color_discrete_map=CORES,
        size='empregos', hover_name='nome_municipio', text='nome_municipio',
        title='Formalizacao x Populacao — RAIS 2022',
        labels={'populacao': 'Populacao', 'pct_formalizacao': '% Formalizacao'}
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(template='plotly_white', height=550, yaxis_ticksuffix='%')
    fig.show()
except KeyError as e:
    print(f'TABELA NAO ENCONTRADA:\n{e}')

---

## Tabelas Gold (disponíveis após rodar notebooks no Fabric)

As queries abaixo só funcionam depois que os seguintes notebooks forem executados no workspace do Fabric:

| Notebook | Tabela criada |
|---|---|
| `nb_gold_mercado_trabalho` | `gold_mercado_trabalho` |
| `nb_gold_populacao` | `gold_populacao_municipios` |
| `nb_gold_pib` | `gold_pib_municipios` |

### Gold: Formalização — `gold_mercado_trabalho` × `gold_populacao_municipios`

In [ ]:
try:
    mt  = T('gold_mercado_trabalho')
    pop = T('gold_populacao_municipios')
    df_f_gold = sql(f"""
        WITH r AS (
            SELECT id_municipio, nome_municipio, cluster,
                   SUM(CAST(vinculos_ativos AS FLOAT)) AS vinculos
            FROM {mt}
            WHERE fonte = 'RAIS' AND ano = 2022
            GROUP BY id_municipio, nome_municipio, cluster
        ),
        p AS (
            SELECT id_municipio,
                   SUM(TRY_CAST(valor AS FLOAT)) AS populacao
            FROM {pop}
            WHERE ano = 2022
            GROUP BY id_municipio
        )
        SELECT r.nome_municipio, r.cluster,
               r.vinculos, p.populacao,
               ROUND((r.vinculos / NULLIF(p.populacao,0)) * 100, 2) AS pct_formalizacao
        FROM r JOIN p ON r.id_municipio = p.id_municipio
        ORDER BY pct_formalizacao DESC
    """)
    display(df_f_gold)
    fig = px.bar(
        df_f_gold.sort_values('pct_formalizacao'),
        x='pct_formalizacao', y='nome_municipio',
        color='cluster', color_discrete_map=CORES, orientation='h',
        text='pct_formalizacao',
        title='(Gold) Taxa de Formalizacao por Municipio — 2022',
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_white', xaxis_ticksuffix='%', height=600)
    fig.show()
except KeyError as e:
    print(f'Gold nao disponivel: {e}')

### Gold: Saldo CAGED — `gold_mercado_trabalho` (2020–2025)

In [ ]:
try:
    mt = T('gold_mercado_trabalho')
    df_c_gold = sql(f"""
        SELECT cluster, ano,
               SUM(CAST(saldo_mensal AS FLOAT)) AS saldo_anual
        FROM {mt}
        WHERE fonte = 'CAGED' AND ano BETWEEN 2020 AND 2025
        GROUP BY cluster, ano ORDER BY cluster, ano
    """)
    display(df_c_gold)
    fig = px.line(
        df_c_gold, x='ano', y='saldo_anual', color='cluster',
        color_discrete_map=CORES, markers=True,
        title='(Gold) Saldo CAGED por Cluster — 2020-2025'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()
except KeyError as e:
    print(f'Gold nao disponivel: {e}')